## Setup

This notebook covers how to **compose agents and manage long-running state**:

- Multi-agent delegation — parent calls specialists, token usage rolls up.
- `pydantic_graph` — explicit workflow modelling with state and branches.
- `agent.iter()` — node-by-node inspection of the agent loop.
- `message_history=` — multi-turn conversations and `new_messages()` vs `all_messages()`.
- Three portable history processors — `sliding_window`, `token_budget_trim`, `summarize_old`.
- `AnthropicCompaction()` — server-side compaction.
- `MemoryTool` — cross-conversation memory revisited

In [2]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Annotated, Literal

from pydantic import BaseModel, Field, field_validator, model_validator
from pydantic_ai.models.anthropic import AnthropicModel
from pydantic_ai.providers.anthropic import AnthropicProvider
from pydantic_settings import BaseSettings, SettingsConfigDict
from rich import print as rprint
from rich.markdown import Markdown

PROJECT_ROOT_PATH = Path.cwd().parent.parent


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=PROJECT_ROOT_PATH / ".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )

    anthropic_api_key: str = Field(...)
    model_id: str = Field(default="claude-haiku-4-5-20251001")


settings = Settings()  # type: ignore[call-arg]

model = AnthropicModel(
    settings.model_id,
    provider=AnthropicProvider(api_key=settings.anthropic_api_key),
)


class Citation(BaseModel):
    doc_id: Annotated[str, Field(min_length=1, max_length=100)]
    quote: Annotated[str, Field(min_length=1, max_length=500)]


class Answer(BaseModel):
    text: Annotated[str, Field(min_length=1, max_length=4000)]
    citations: list[Citation] = Field(default_factory=list)
    confidence: Annotated[float, Field(ge=0.0, le=1.0)]
    risk_flag: Literal["none", "escalate", "urgent"] = "none"

    @field_validator("citations")
    @classmethod
    def reject_empty_citation_quotes(cls, v: list[Citation]) -> list[Citation]:
        if any(not c.quote.strip() for c in v):
            raise ValueError("citation quotes must not be empty or whitespace")
        return v

    @model_validator(mode="after")
    def low_confidence_must_admit_uncertainty(self) -> "Answer":
        t = self.text.lower()
        if self.confidence < 0.4 and "not sure" not in t and "may" not in t:
            raise ValueError(
                'confidence < 0.4 — text must signal uncertainty (e.g. "not sure"/"may")'
            )
        return self


rprint(Markdown("# Workflows, state, and memory"))
rprint("model_id:", settings.model_id)

Workflows, state, and memory

model_id: claude-haiku-4-5-20251001

## Multi-agent delegation

A *router* agent classifies the question and hands off to one of three specialists via a tool. `usage=ctx.usage` on the nested `.run()` call ensures every child's token usage rolls up into the parent's `result.usage()`, so you get one total cost across the whole chain.

Three specialists in the example:

- **`product_specialist`** — plans, features, pricing.
- **`support_specialist`** — general help, returns, complaints.
- **`billing_specialist`** — invoices, payment methods, subscription changes.

…and a `router_agent` with one `dispatch(category, question)` tool that picks the right one.

In [3]:
from pydantic_ai import Agent, RunContext

# Compact catalogue the product specialist quotes from.
_CATALOGUE_TEXT = (
    "Catalogue (USD/month):\n"
    "- basic_plan: $9.99 (1 user)\n"
    "- pro_plan: $29.99 (10 users)\n"
    "- enterprise_plan: $99.99 (unlimited)\n"
    "- addon_storage: $4.99\n"
    "- addon_priority_support: $19.99"
)

# ----- the three specialists -----
product_specialist = Agent[str, Answer](
    model,
    output_type=Answer,
    deps_type=str,
    instructions=(
        "You answer questions about plans, features, and pricing. "
        "Quote concrete numbers from the catalogue below. Always return the "
        "Answer schema. If unsure, set confidence below 0.4 and say so.\n\n"
        + _CATALOGUE_TEXT
    ),
)

support_specialist = Agent[str, Answer](
    model,
    output_type=Answer,
    deps_type=str,
    instructions=(
        "You handle general support: returns, complaints, how-to questions. "
        "If a question seems sensitive or out of scope, set risk_flag='escalate'. "
        "Always return the Answer schema."
    ),
)

billing_specialist = Agent[str, Answer](
    model,
    output_type=Answer,
    deps_type=str,
    instructions=(
        "You answer questions about invoices, payment methods, and subscription "
        "changes. You do NOT quote prices — that's the product specialist's job. "
        "If a billing issue looks urgent (failed charges, double-billing), "
        "set risk_flag='urgent'. Always return the Answer schema."
    ),
)


# ----- the router with a dispatch tool -----
router_agent = Agent[str, Answer](
    model,
    output_type=Answer,
    deps_type=str,
    instructions=(
        "You are the router. Call the `dispatch` tool exactly once with the "
        "right category and the question text. Categories:\n"
        "- 'product': plans, features, pricing, what's on offer.\n"
        "- 'support': help, returns, complaints, how-to questions.\n"
        "- 'billing': invoices, payment methods, subscription changes "
        "(NOT pricing — that's product).\n"
        "- 'general': greetings or smalltalk — answer yourself with low confidence."
    ),
)


@router_agent.tool
async def dispatch(
    ctx: RunContext[str],
    category: Literal["product", "support", "billing", "general"],
    question: str,
) -> Answer:
    """Hand the question to the right specialist. Token usage rolls up via ctx.usage."""
    specialist = {
        "product": product_specialist,
        "support": support_specialist,
        "billing": billing_specialist,
    }.get(category)

    if specialist is None:
        return Answer(
            text="I'm not sure which specialist should take this — can you give more context?",
            confidence=0.3,
        )
    # usage=ctx.usage is the key bit: child tokens accumulate into the parent's counter.
    result = await specialist.run(question, deps=ctx.deps, usage=ctx.usage)
    return result.output


# ----- run it -----
result = await router_agent.run("How much does the pro_plan cost per month?", deps="Alice")
rprint("answer.text       :", result.output.text[:200])
rprint("answer.confidence :", result.output.confidence)
rprint("answer.risk_flag  :", result.output.risk_flag)

rprint("\n--- rolled-up usage (parent + specialist call) ---")
rprint(result.usage())

rprint("\n--- tool calls made by the router ---")
for msg in result.new_messages():
    for part in getattr(msg, "parts", []) or []:
        if part.__class__.__name__ == "ToolCallPart":
            rprint(f"  {part.tool_name}({part.args!r})")

answer.text       : The pro_plan costs $29.99 per month and includes support for up to 10 users.

answer.confidence : 1.0

answer.risk_flag  : none

--- rolled-up usage (parent + specialist call) ---

RunUsage(input_tokens=3194, output_tokens=310, details={'input_tokens': 3194, 'output_tokens': 310, 
'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0}, requests=3, tool_calls=1)

--- tool calls made by the router ---

dispatch({'category': 'product', 'question': 'How much does the pro_plan cost per month?'})

final_result({'text': 'The pro_plan costs $29.99 per month and includes support for up to 10 users.', 
'confidence': 1.0, 'citations': [{'doc_id': 'pricing_catalogue', 'quote': 'pro_plan: $29.99 (10 users)'}]})

## `pydantic_graph` — explicit workflow modelling

Once a flow has branches, retries, and shared mutable state, a plain agent chain hides the control flow. A graph gives us a way to make a custom (and predictable) pipeline.

Workflow for this section:

```
ClassifyNode → RetrieveNode → AnswerNode → CritiqueNode → (loop or End)
```

`CritiqueNode` reviews the draft. If the critic approves (or revisions are exhausted), it returns `End(answer)`. Otherwise it loops back to `AnswerNode` with the feedback appended to state. State is a plain `@dataclass` — Pydantic-graph doesn't require any special base class on it.

In [9]:
from dataclasses import dataclass, field
from pydantic_graph import BaseNode, End, Graph, GraphRunContext

Category = Literal["product", "support", "billing", "general"]


# ----- shared state, threaded through every node -----
@dataclass
class GraphState:
    question: str
    user: str
    category: str | None = None
    retrieved: list[str] = field(default_factory=list)
    draft: Answer | None = None
    revisions_remaining: int = 2
    critique_notes: list[str] = field(default_factory=list)


# ----- helper agents that nodes call -----
_classifier_agent = Agent[None, Category](
    model,
    output_type=Category,
    instructions=(
        "Classify the question into one category: 'product', 'support', 'billing', "
        "or 'general'. Reply with only the category."
    ),
)

_critic_agent = Agent[None, tuple[bool, str]](
    model,
    output_type=tuple[bool, str],
    instructions=(
        "Given a draft Answer JSON and the original question, return (approve, feedback). "
        "Approve=True if on-topic, cites sources where needed, and risk_flag fits. "
        "Otherwise False with short actionable feedback."
    ),
)

_answerer_agent = Agent[str, Answer](
    model,
    output_type=Answer,
    deps_type=str,
    instructions=(
        "Answer the user's question. Use the retrieved snippets when relevant. "
        "Always return the Answer schema."
    ),
)


# ----- a tiny corpus for the retrieve step -----
_CORPUS = {
    "returns": "Returns within 30 days of delivery, no questions asked. Refunds in 5 business days.",
    "shipping": "Standard shipping 3-5 business days, free on orders over $50. Express $12.99 (1-2 days).",
    "billing": "Subscriptions billed monthly on signup date. Cancel any time; cancellation effective at period end.",
    "subscriptions": "basic_plan $9.99/mo (1 user). pro_plan $29.99/mo (10 users). enterprise_plan $99.99/mo (unlimited).",
    "support": "Email support for all customers. Priority phone support for pro and enterprise tiers.",
}


# ----- the nodes -----
@dataclass
class ClassifyNode(BaseNode[GraphState, None, Answer]):
    async def run(self, ctx: GraphRunContext[GraphState]) -> "RetrieveNode":
        result = await _classifier_agent.run(ctx.state.question)
        ctx.state.category = result.output
        return RetrieveNode()


@dataclass
class RetrieveNode(BaseNode[GraphState, None, Answer]):
    """Linear-scan lookup over _CORPUS. (Real version would be BM25 — see 02_capabilities.)"""

    async def run(self, ctx: GraphRunContext[GraphState]) -> "AnswerNode":
        q = ctx.state.question.lower()
        ctx.state.retrieved = [
            f"[{doc_id}] {text}"
            for doc_id, text in _CORPUS.items()
            if any(tok in q for tok in doc_id.split("_")) or doc_id in q
        ] or [f"[{doc_id}] {text}" for doc_id, text in list(_CORPUS.items())[:2]]
        return AnswerNode()


@dataclass
class AnswerNode(BaseNode[GraphState, None, Answer]):
    async def run(self, ctx: GraphRunContext[GraphState]) -> "CritiqueNode":
        parts: list[str] = [f"Question: {ctx.state.question}"]
        if ctx.state.category:
            parts.append(f"Category: {ctx.state.category}")
        if ctx.state.retrieved:
            parts.append("Retrieved snippets:\n" + "\n".join(ctx.state.retrieved))
        if ctx.state.critique_notes:
            parts.append(
                "Reviewer feedback to address:\n- " + "\n- ".join(ctx.state.critique_notes)
            )
        result = await _answerer_agent.run("\n\n".join(parts), deps=ctx.state.user)
        ctx.state.draft = result.output
        return CritiqueNode()


@dataclass
class CritiqueNode(BaseNode[GraphState, None, Answer]):
    async def run(
        self, ctx: GraphRunContext[GraphState]
    ) -> AnswerNode | End[Answer]:
        assert ctx.state.draft is not None
        review_prompt = (
            f"Original question: {ctx.state.question}\n\n"
            f"Draft answer JSON: {ctx.state.draft.model_dump_json()}"
        )
        result = await _critic_agent.run(review_prompt)
        approved, feedback = result.output
        if approved or ctx.state.revisions_remaining == 0:
            return End(ctx.state.draft)
        ctx.state.revisions_remaining -= 1
        if feedback:
            ctx.state.critique_notes.append(feedback)
        return AnswerNode()


GRAPH_NODES = [ClassifyNode, RetrieveNode, AnswerNode, CritiqueNode]
graph: Graph[GraphState, None, Answer] = Graph(nodes=GRAPH_NODES)

print("graph wired:", [n.__name__ for n in GRAPH_NODES])

graph wired: ['ClassifyNode', 'RetrieveNode', 'AnswerNode', 'CritiqueNode']


In [5]:
# pydantic_graph emits Mermaid source for the wired-up nodes — paste into
# https://mermaid.live to see the flow.
try:
    print(graph.mermaid_code(start_node=ClassifyNode))
except Exception as exc:  # noqa: BLE001
    print(f"(mermaid_code unavailable: {exc!r})")

# Run the graph end-to-end.
state = GraphState(
    question="How long is the return window?",
    user="Alice",
)
run_result = await graph.run(ClassifyNode(), state=state)
answer = run_result.output

rprint("\n--- final answer ---")
rprint("text       :", answer.text[:240])
rprint("confidence :", answer.confidence)
rprint("risk_flag  :", answer.risk_flag)
rprint("category   :", state.category)
rprint("revisions  :", 2 - state.revisions_remaining, "of 2 used")

---
title: graph
---
stateDiagram-v2
  [*] --> ClassifyNode
  ClassifyNode --> RetrieveNode
  RetrieveNode --> AnswerNode
  AnswerNode --> CritiqueNode
  CritiqueNode --> AnswerNode
  CritiqueNode --> [*]


--- final answer ---

text       : The return window is 30 days from delivery. Returns are accepted with no questions asked, and refunds 
are processed within 5 business days.

confidence : 0.95

risk_flag  : none

category   : billing

revisions  : 0 of 2 used

In [10]:
for message in result.all_messages():
    rprint(message)

ModelRequest(parts=[UserPromptPart(content='How much does the pro_plan cost per month?', 
timestamp=datetime.datetime(2026, 5, 20, 5, 54, 15, 73726, tzinfo=datetime.timezone.utc))], 
timestamp=datetime.datetime(2026, 5, 20, 5, 54, 15, 73914, tzinfo=datetime.timezone.utc), instructions="You are the
router. Call the `dispatch` tool exactly once with the right category and the question text. Categories:\n- 
'product': plans, features, pricing, what's on offer.\n- 'support': help, returns, complaints, how-to questions.\n-
'billing': invoices, payment methods, subscription changes (NOT pricing — that's product).\n- 'general': greetings 
or smalltalk — answer yourself with low confidence.", run_id='019e43f2-ef9c-743b-9e7f-7657ba6759a5', 
conversation_id='019e43f2-ef9c-743b-9e7f-76567dbcfc9c')

ModelResponse(parts=[ToolCallPart(tool_name='dispatch', args={'category': 'product', 'question': 'How much does the
pro_plan cost per month?'}, tool_call_id='toolu_01QHsNr5Jep5sC7UqiPQzmTu')], usage=RequestUsage(input_tokens=1038, 
output_tokens=63, details={'input_tokens': 1038, 'output_tokens': 63, 'cache_creation_input_tokens': 0, 
'cache_read_input_tokens': 0}), model_name='claude-haiku-4-5-20251001', timestamp=datetime.datetime(2026, 5, 20, 5,
54, 15, 988499, tzinfo=datetime.timezone.utc), provider_name='anthropic', provider_url='https://api.anthropic.com',
provider_details={'finish_reason': 'tool_use'}, provider_response_id='msg_01L7qAFKr2792LW7rR38JmVj', 
finish_reason='tool_call', run_id='019e43f2-ef9c-743b-9e7f-7657ba6759a5', 
conversation_id='019e43f2-ef9c-743b-9e7f-76567dbcfc9c')

ModelRequest(parts=[ToolReturnPart(tool_name='dispatch', content=Answer(text='The pro_plan costs $29.99 per month 
and includes support for up to 10 users.', citations=[Citation(doc_id='pricing_catalogue', quote='pro_plan: $29.99 
(10 users)')], confidence=1.0, risk_flag='none'), tool_call_id='toolu_01QHsNr5Jep5sC7UqiPQzmTu', 
timestamp=datetime.datetime(2026, 5, 20, 5, 54, 17, 86176, tzinfo=datetime.timezone.utc))], 
timestamp=datetime.datetime(2026, 5, 20, 5, 54, 17, 86625, tzinfo=datetime.timezone.utc), instructions="You are the
router. Call the `dispatch` tool exactly once with the right category and the question text. Categories:\n- 
'product': plans, features, pricing, what's on offer.\n- 'support': help, returns, complaints, how-to questions.\n-
'billing': invoices, payment methods, subscription changes (NOT pricing — that's product).\n- 'general': greetings 
or smalltalk — answer yourself with low confidence.", run_id='019e43f2-ef9c-743b-9e7f-7657ba6759a5', 
conversation_id='019e43f2-ef9c-743b-9e7f-76567dbcfc9c')

ModelResponse(parts=[ToolCallPart(tool_name='final_result', args={'text': 'The pro_plan costs $29.99 per month and 
includes support for up to 10 users.', 'confidence': 1.0, 'citations': [{'doc_id': 'pricing_catalogue', 'quote': 
'pro_plan: $29.99 (10 users)'}]}, tool_call_id='toolu_019Mb3mpZu5Br5NQR5nuQpHj')], 
usage=RequestUsage(input_tokens=1198, output_tokens=122, details={'input_tokens': 1198, 'output_tokens': 122, 
'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0}), model_name='claude-haiku-4-5-20251001', 
timestamp=datetime.datetime(2026, 5, 20, 5, 54, 26, 595929, tzinfo=datetime.timezone.utc), 
provider_name='anthropic', provider_url='https://api.anthropic.com', provider_details={'finish_reason': 
'tool_use'}, provider_response_id='msg_014DFJh8DaNwxZwEzJZYVENZ', finish_reason='tool_call', 
run_id='019e43f2-ef9c-743b-9e7f-7657ba6759a5', conversation_id='019e43f2-ef9c-743b-9e7f-76567dbcfc9c')

ModelRequest(parts=[ToolReturnPart(tool_name='final_result', content='Final result processed.', 
tool_call_id='toolu_019Mb3mpZu5Br5NQR5nuQpHj', timestamp=datetime.datetime(2026, 5, 20, 5, 54, 26, 596946, 
tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 20, 5, 54, 26, 596996, 
tzinfo=datetime.timezone.utc), run_id='019e43f2-ef9c-743b-9e7f-7657ba6759a5', 
conversation_id='019e43f2-ef9c-743b-9e7f-76567dbcfc9c')

## `agent.iter()` — step through an agent loop

`agent.run()` hides the agent's internal loop. `agent.iter()` yields each node — `UserPromptNode`, `ModelRequestNode`, `CallToolsNode`, etc.

The example builds a small agent with one function tool (`lookup_price`) so the loop has a tool-call node to walk past.

In [11]:
_PRICE_LIST = {
    "basic_plan": 9.99,
    "pro_plan": 29.99,
    "enterprise_plan": 99.99,
    "addon_storage": 4.99,
    "addon_priority_support": 19.99,
}


def lookup_price(item_name: str) -> dict:
    """Look up a product price by canonical name."""
    key = item_name.strip().lower()
    if key not in _PRICE_LIST:
        return {"error": "unknown item", "known": sorted(_PRICE_LIST)}
    return {"item_name": key, "price_usd": _PRICE_LIST[key]}


iter_agent = Agent[None, Answer](
    model,
    output_type=Answer,
    instructions=(
        "Use the lookup_price tool when the user asks about a specific product price. "
        "Always answer in the Answer format and cite doc_id='pricing' for any fee figure."
    ),
    tools=[lookup_price],
)

async with iter_agent.iter("What does the pro_plan cost?") as agent_run:
    step = 0
    async for node in agent_run:
        step += 1
        print(f"step {step:02d}: {type(node).__name__}")

final = agent_run.result
rprint("\n--- final ---")
rprint("answer    :", final.output.text[:160])
rprint("confidence:", final.output.confidence)
rprint("usage     :", agent_run.usage())

step 01: UserPromptNode
step 02: ModelRequestNode
step 03: CallToolsNode
step 04: ModelRequestNode
step 05: CallToolsNode
step 06: End


--- final ---

answer    : The pro_plan costs $29.99 USD.

confidence: 0.95

usage     :
RunUsage(input_tokens=1966, output_tokens=143, details={'input_tokens': 1966, 'output_tokens': 143, 
'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0}, requests=2, tool_calls=1)

## `message_history=` — multi-turn conversations

To make the agent remember the previous turn, pass the prior `result.all_messages()` into the next `agent.run(...)`. Pydantic-ai prepends it to the new prompt for you. Two distinctions to know:

- **`result.new_messages()`** — only what *this* turn added (the new user prompt + the model's reply, tool calls, tool returns).
- **`result.all_messages()`** — the full transcript including everything passed in via `message_history=`.

Use `all_messages()` when you persist the rolling conversation; use `new_messages()` when you append a delta to your own store.

In [12]:
multi_turn_agent = Agent[str, Answer](
    model,
    output_type=Answer,
    deps_type=str,
    instructions=(
        "Be concise. Always return the Answer schema. Use the conversation history "
        "to track what product the user is asking about."
    ),
)

# Turn 1 — establish what we're talking about.
r1 = await multi_turn_agent.run("I'm interested in the pro_plan.", deps="Alice")
rprint("turn 1:", r1.output.text[:200])

# Turn 2 — the model references turn 1's product without us re-stating it.
r2 = await multi_turn_agent.run(
    "What's the price again?",
    deps="Alice",
    message_history=r1.all_messages(),
)
rprint("\nturn 2:", r2.output.text[:200])

# Inspect the message-list distinction.
rprint("\nlen(r2.new_messages()) :", len(r2.new_messages()), "  # just turn 2's additions")
rprint("len(r2.all_messages()) :", len(r2.all_messages()), "  # whole transcript")
rprint(
    "delta added by turn 2  :",
    len(r2.all_messages()) - len(r1.all_messages()),
    "  # turn 2 contribution to the rolling transcript",
)

turn 1: I'd be happy to help you with information about the Pro Plan! However, I need a bit more context to provide
you with accurate details. Could you please specify what you'd like to know about the Pro Pl

turn 2: I don't have access to specific pricing information for the Pro Plan in my current knowledge base. To get 
accurate pricing details, I recommend:

1. Checking the product's official website or pricing

len(r2.new_messages()) : 3   # just turn 2's additions

len(r2.all_messages()) : 6   # whole transcript

delta added by turn 2  : 3   # turn 2 contribution to the rolling transcript

In [14]:
for i, msg in enumerate(r2.all_messages(), start=1):
    rprint(f"message {i:02d}:", msg)

message 01:
ModelRequest(parts=[UserPromptPart(content="I'm interested in the pro_plan.", timestamp=datetime.datetime(2026, 5, 
20, 6, 0, 43, 48204, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 20, 6, 0, 43, 48568, 
tzinfo=datetime.timezone.utc), instructions='Be concise. Always return the Answer schema. Use the conversation 
history to track what product the user is asking about.', run_id='019e43f8-db26-733c-ab64-ecc8c4b7c8da', 
conversation_id='019e43f8-db26-733c-ab64-ecc7e3da34ea')

message 02:
ModelResponse(parts=[ToolCallPart(tool_name='final_result', args={'text': "I'd be happy to help you with 
information about the Pro Plan! However, I need a bit more context to provide you with accurate details. Could you 
please specify what you'd like to know about the Pro Plan? For example:\n\n- What are the features included?\n- 
What is the pricing?\n- How does it compare to other plans?\n- What are the terms and conditions?\n- Is there 
anything specific about the Pro Plan you're concerned about?\n\nOnce you provide more details about what you're 
looking for, I can give you a comprehensive answer.", 'confidence': 0.5}, 
tool_call_id='toolu_01GT6pZVTpreMjLeKp7njy9v')], usage=RequestUsage(input_tokens=863, output_tokens=172, 
details={'input_tokens': 863, 'output_tokens': 172, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 
0}), model_name='claude-haiku-4-5-20251001', timestamp=datetime.datetime(2026, 5, 20, 6, 0, 45, 364477, 
tzinfo=datetime.timezone.utc), provider_name='anthropic', provider_url='https://api.anthropic.com', 
provider_details={'finish_reason': 'tool_use'}, provider_response_id='msg_018T8hSvK6x85SrKfuaEqbBG', 
finish_reason='tool_call', run_id='019e43f8-db26-733c-ab64-ecc8c4b7c8da', 
conversation_id='019e43f8-db26-733c-ab64-ecc7e3da34ea')

message 03:
ModelRequest(parts=[ToolReturnPart(tool_name='final_result', content='Final result processed.', 
tool_call_id='toolu_01GT6pZVTpreMjLeKp7njy9v', timestamp=datetime.datetime(2026, 5, 20, 6, 0, 45, 365453, 
tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 20, 6, 0, 45, 365492, 
tzinfo=datetime.timezone.utc), run_id='019e43f8-db26-733c-ab64-ecc8c4b7c8da', 
conversation_id='019e43f8-db26-733c-ab64-ecc7e3da34ea')

message 04:
ModelRequest(parts=[UserPromptPart(content="What's the price again?", timestamp=datetime.datetime(2026, 5, 20, 6, 
0, 45, 371234, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 20, 6, 0, 45, 371509, 
tzinfo=datetime.timezone.utc), instructions='Be concise. Always return the Answer schema. Use the conversation 
history to track what product the user is asking about.', run_id='019e43f8-e439-75c6-88d5-e8c86a43e101', 
conversation_id='019e43f8-db26-733c-ab64-ecc7e3da34ea')

message 05:
ModelResponse(parts=[ToolCallPart(tool_name='final_result', args={'text': "I don't have access to specific pricing 
information for the Pro Plan in my current knowledge base. To get accurate pricing details, I recommend:\n\n1. 
Checking the product's official website or pricing page\n2. Contacting the sales or support team directly\n3. 
Reviewing any documentation or materials you may have received\n\nCould you provide me with any pricing information
you've already seen, or let me know if you have a specific document or source where this information is located? 
That way I can help you better understand the pricing structure.", 'confidence': 0.3}, 
tool_call_id='toolu_01KKMUdYSUjyuBSP6zfUSysr')], usage=RequestUsage(input_tokens=1077, output_tokens=165, 
details={'input_tokens': 1077, 'output_tokens': 165, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 
0}), model_name='claude-haiku-4-5-20251001', timestamp=datetime.datetime(2026, 5, 20, 6, 0, 48, 717771, 
tzinfo=datetime.timezone.utc), provider_name='anthropic', provider_url='https://api.anthropic.com', 
provider_details={'finish_reason': 'tool_use'}, provider_response_id='msg_012zabBBq6JnMnNB9hLEHV5a', 
finish_reason='tool_call', run_id='019e43f8-e439-75c6-88d5-e8c86a43e101', 
conversation_id='019e43f8-db26-733c-ab64-ecc7e3da34ea')

message 06:
ModelRequest(parts=[ToolReturnPart(tool_name='final_result', content='Final result processed.', 
tool_call_id='toolu_01KKMUdYSUjyuBSP6zfUSysr', timestamp=datetime.datetime(2026, 5, 20, 6, 0, 48, 718651, 
tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 20, 6, 0, 48, 718690, 
tzinfo=datetime.timezone.utc), run_id='019e43f8-e439-75c6-88d5-e8c86a43e101', 
conversation_id='019e43f8-db26-733c-ab64-ecc7e3da34ea')

## `sliding_window(n)` — cheapest history processor

Keep only the last `n` messages. Free, deterministic, lossy. Anything older than `n` is dropped.

A `history_processor` is just an async function `list[ModelMessage] -> list[ModelMessage]` that pydantic-ai applies before every model call. You pass them via `Agent(..., history_processors=[...])`.

In [15]:
from collections.abc import Awaitable, Callable

from pydantic_ai.messages import ModelMessage

Processor = Callable[[list[ModelMessage]], Awaitable[list[ModelMessage]]]


def sliding_window(n: int) -> Processor:
    """Keep only the last `n` messages."""

    async def _processor(messages: list[ModelMessage]) -> list[ModelMessage]:
        return messages[-n:] if len(messages) > n else messages

    return _processor


# Wire into an agent and run a 5-turn conversation. With window=4, turn 5's
# request can no longer see the first turn's "remember my name is Alice".
windowed_agent = Agent[None, str](
    model,
    output_type=str,
    instructions="Be concise.",
    history_processors=[sliding_window(4)],
)

res = await windowed_agent.run("hi, my name is Alice — remember it")
res = await windowed_agent.run("what's 1+1?", message_history=res.all_messages())
res = await windowed_agent.run("what's 2+2?", message_history=res.all_messages())
res = await windowed_agent.run("what's 3+3?", message_history=res.all_messages())
res = await windowed_agent.run("what's my name?", message_history=res.all_messages())
rprint("final answer:", res.output[:200])
rprint(
    "\nObserve: the agent likely can't recall 'Alice' — that turn fell out of the window."
)

final answer: I don't know your name. You haven't told me what it is. Feel free to share if you'd like!

Observe: the agent likely can't recall 'Alice' — that turn fell out of the window.

## `token_budget_trim(budget)` — history management based on the number of tokens

Drops oldest messages until the running token total fits under `budget`. Uses `tiktoken`'s `cl100k_base` as a proxy for Anthropic tokens.

Useful when you want a *size* guarantee rather than a *count* guarantee — chats with a few very long messages vs many short ones behave differently under `sliding_window` but identically under `token_budget_trim`.

In [16]:
import tiktoken

from pydantic_ai.messages import ModelRequest, UserPromptPart


def token_budget_trim(budget: int, encoding: str = "cl100k_base") -> Processor:
    """Drop oldest messages until total tokens fit under `budget`."""
    enc = tiktoken.get_encoding(encoding)

    def _count(message: ModelMessage) -> int:
        try:
            payload = message.model_dump_json()  # type: ignore[attr-defined]
        except AttributeError:
            payload = repr(message)
        return len(enc.encode(payload))

    async def _processor(messages: list[ModelMessage]) -> list[ModelMessage]:
        kept: list[ModelMessage] = []
        running = 0
        for message in reversed(messages):
            cost = _count(message)
            if running + cost > budget and kept:
                break
            kept.append(message)
            running += cost
        kept.reverse()
        return kept

    return _processor


# Smoke test on a fake history — 5 user prompts, each ~30 tokens.
fake_history: list[ModelMessage] = [
    ModelRequest(parts=[UserPromptPart(content=f"turn {i}: " + "subscription question " * 6)])
    for i in range(5)
]
trimmer = token_budget_trim(budget=120)
kept = await trimmer(fake_history)
rprint(f"input messages : {len(fake_history)}")
rprint(f"kept messages  : {len(kept)}  (those that fit under budget=120)")

# And wired into an agent.
budget_agent = Agent[None, str](
    model,
    output_type=str,
    instructions="Reply in one short sentence.",
    history_processors=[token_budget_trim(budget=2000)],
)
out = await budget_agent.run("What's the difference between basic_plan and pro_plan?")
rprint("\nanswer :", out.output[:200])
rprint("usage  :", out.usage())

input messages : 5

kept messages  : 2  (those that fit under budget=120)

answer : I don't have information about what "basic_plan" and "pro_plan" refer to without more context—could you 
clarify what product or service you're asking about?

usage  :
RunUsage(input_tokens=26, output_tokens=40, details={'input_tokens': 26, 'output_tokens': 40, 
'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0}, requests=1)

## `summarize_old(threshold, model)` — LLM summarisation

Once history grows past `threshold` messages, summarise the *oldest half* into one synthetic user-prompt message; keep the rest verbatim. A cheap model does the distillation. Results are cached by a hash of the summarised prefix — same prefix, no extra LLM call on re-run.

Cost: one extra LLM call *per compaction event*. Compared to `sliding_window`, this keeps the *gist* of old turns instead of dropping them outright.

In [ ]:
import hashlib

_summary_cache: dict[str, str] = {}


def _signature(messages: list[ModelMessage]) -> str:
    """Stable SHA-256 hash so identical prefixes hit the cache."""
    h = hashlib.sha256()
    for message in messages:
        try:
            h.update(message.model_dump_json().encode("utf-8"))  # type: ignore[attr-defined]
        except AttributeError:
            h.update(repr(message).encode("utf-8"))
        h.update(b"\x00")
    return h.hexdigest()


def summarize_old(threshold: int, summary_model: AnthropicModel) -> Processor:
    """Summarise the oldest half of the conversation when it grows past `threshold`."""
    summary_agent: Agent[None, str] = Agent[None, str](
        summary_model,
        output_type=str,
        instructions=(
            "Summarise the following conversation excerpt into one tight paragraph. "
            "Preserve facts that may be referenced later. Do not add commentary."
        ),
    )

    async def _processor(messages: list[ModelMessage]) -> list[ModelMessage]:
        if len(messages) <= threshold:
            return messages
        cut = len(messages) // 2
        old, recent = messages[:cut], messages[cut:]
        sig = _signature(old)
        summary = _summary_cache.get(sig)
        if summary is None:
            transcript = "\n".join(
                getattr(m, "model_dump_json", lambda: repr(m))()
                for m in old
            )
            result = await summary_agent.run(transcript)
            summary = result.output
            _summary_cache[sig] = summary
        synthetic = ModelRequest(
            parts=[UserPromptPart(content=f"Earlier conversation summary: {summary}")]
        )
        return [synthetic, *recent]

    return _processor


# Low threshold so compaction fires within a handful of turns.
summary_demo_agent = Agent[None, str](
    model,
    output_type=str,
    instructions="Be brief.",
    history_processors=[summarize_old(threshold=4, summary_model=model)],
)

history: list[ModelMessage] = []
questions = [
    "What's in the basic_plan?",
    "How long does standard shipping take?",
    "Can I cancel any time?",
    "What's priority support?",
    "Compare pro_plan and enterprise_plan in one sentence.",
]
for q in questions:
    res = await summary_demo_agent.run(q, message_history=history)
    history = res.all_messages()

rprint("final answer :", res.output[:240])
rprint(f"\nfinal history length: {len(history)} messages")
rprint(f"summary cache entries : {len(_summary_cache)}  (one per compaction event)")

## `AnthropicCompaction()` — provider-native compaction

Anthropic compacts server-side once input tokens cross `token_threshold`. You don't pay for the summariser — the model does it inline. The trade-off vs the portable processors: cheaper and zero code on your side, but Anthropic-specific and opaque (you can't audit what was dropped).

Watch `result.usage().details` for compaction-related counters when it kicks in.

**Model gating.** The context-management beta is currently only available on Sonnet / Opus tier models — Haiku will reject the request with a 400. The cell below builds a Sonnet model just for this section. If your key doesn't have Sonnet access, swap the model id to whatever does.

We build two agents (`plain` + `compacted`), feed them the same synthetic 30-turn history, and compare `usage()`.

In [ ]:
from pydantic_ai.messages import ModelResponse, TextPart
from pydantic_ai.models.anthropic import AnthropicCompaction

# AnthropicCompaction is gated to certain models — Haiku 4.5 rejects it.
compaction_model = AnthropicModel(
    "claude-sonnet-4-6",
    provider=AnthropicProvider(api_key=settings.anthropic_api_key),
)


def _synthesise_history(turns: int = 30) -> list[ModelMessage]:
    """Build a fictional support transcript of `turns` user/assistant pairs."""
    topics = [
        "the pro_plan vs enterprise_plan tradeoffs",
        "shipping options and timelines",
        "the cancellation policy and refund timing",
        "priority support coverage",
        "addon_storage and addon_priority_support pricing",
    ]
    history: list[ModelMessage] = []
    for i in range(turns):
        topic = topics[i % len(topics)]
        history.append(
            ModelRequest(
                parts=[
                    UserPromptPart(
                        content=(
                            f"[turn {i}] Walk me through {topic} in detail. I want concrete "
                            "numbers, every exception, and a comparison to common alternatives "
                            "so I can make an informed decision."
                        )
                    )
                ]
            )
        )
        history.append(
            ModelResponse(
                parts=[
                    TextPart(
                        content=(
                            f"[turn {i}] On {topic}: pro_plan is $29.99/mo for up to 10 users "
                            "with priority phone support. enterprise_plan is $99.99/mo with "
                            "unlimited users and a dedicated success manager. Standard shipping "
                            "is free over $50 and arrives in 3-5 business days; express is "
                            "$12.99 and arrives in 1-2 days. Cancellation is effective at the "
                            "end of the current billing period; refunds in 5 business days."
                        )
                    )
                ]
            )
        )
    return history


synth_history = _synthesise_history(turns=350)
rprint(f"synthesised history: {len(synth_history)} messages ({len(synth_history) // 2} turns)")

plain = Agent[None, str](
    compaction_model,
    output_type=str,
    instructions="Be concise.",
)
compacted = Agent[None, str](
    compaction_model,
    output_type=str,
    instructions="Be concise.",
    capabilities=[AnthropicCompaction(token_threshold=50_000)],
)

new_prompt = "Quick refresher: what's the pro_plan price again?"

rprint("\n--- plain agent (no compaction) ---")
plain_result = await plain.run(new_prompt, message_history=synth_history)
rprint("answer :", plain_result.output[:160])
rprint("usage  :", plain_result.usage())

rprint("\n--- compacted agent (AnthropicCompaction(token_threshold=50_000)) ---")
compacted_result = await compacted.run(new_prompt, message_history=synth_history)
rprint("answer :", compacted_result.output[:160])
rprint("usage  :", compacted_result.usage())
rprint(
    "\nObserve: compacted usage carries `compaction_*` keys when it fires; "
    "plain does not."
)

## `MemoryTool` — cross-conversation memory

Anthropic's MemoryTool is filesystem-shaped — the model issues `view` / `create` / `str_replace` / `delete` commands against paths under `/memories/`. The backing function tool has to dispatch on `command` and act on whatever store you give it. In production that's SQLite / Redis / a file.

Two turns, on purpose set up as *independent* runs (no `message_history=`):

1. Turn 1 — the model stores a preference.
2. Turn 2 — fresh run, no message history. The model reads `/memories/` autonomously and answers using what it wrote earlier.

The continuity comes entirely from `_memory_store`, not from the conversation transcript.

In [ ]:
from pydantic_ai.builtin_tools import MemoryTool

# In-process storage. Swap for a SQLite table / Redis / file for real persistence.
_memory_store: dict[str, str] = {}


def memory(command: str, path: str, file_text: str = "", **kwargs) -> str:
    """Backing storage for the Anthropic MemoryTool builtin.

    Dispatches on `command`: view, create, str_replace, delete.
    """
    if command == "view":
        if path.rstrip("/") in ("/memories", ""):
            files = sorted(_memory_store)
            return "\n".join(files) if files else "(no memories yet)"
        return _memory_store.get(path, f"(not found: {path})")

    if command == "create":
        _memory_store[path] = file_text
        return f"created {path}"

    if command == "str_replace":
        old, new = kwargs.get("old_str", ""), kwargs.get("new_str", "")
        if path not in _memory_store:
            return f"not found: {path}"
        _memory_store[path] = _memory_store[path].replace(old, new)
        return f"updated {path}"

    if command == "delete":
        _memory_store.pop(path, None)
        return f"deleted {path}"

    return f"unsupported command: {command}"


memory_agent = Agent[None, str](
    model,
    output_type=str,
    builtin_tools=[MemoryTool()],
    tools=[memory],  # name must be 'memory' — that's the gotcha from 02_capabilities.
    instructions=(
        "You are a helpful assistant with persistent memory. When the user shares a "
        "preference or fact about themselves, store it under /memories/ using the "
        "memory tool. When asked a question, first check /memories/ for relevant "
        "context before answering."
    ),
)

# Turn 1 — model stores something.
r1 = await memory_agent.run(
    "I prefer the pro_plan and I dislike email notifications. Please remember this."
)
rprint("--- turn 1 ---")
rprint(r1.output[:240])
rprint("\nstore after turn 1:", _memory_store)

# Turn 2 — FRESH run. No message_history. Model reads /memories/ autonomously.
r2 = await memory_agent.run(
    "Based on what you know about me, what should I look at, and how should we contact me?"
)
rprint("\n--- turn 2 (fresh run, no message_history) ---")
rprint(r2.output[:300])

## Decision matrix — portable vs server-side vs memory

Pick the row that matches your situation.

| Use case                                                              | Best fit                                       |
|-----------------------------------------------------------------------|------------------------------------------------|
| Demo / prototype, ≤ 30 turns                                          | Do nothing.                                    |
| Anthropic, long chats, no special needs                               | `AnthropicCompaction(token_threshold=100_000)` |
| Anthropic, agent should remember user facts across sessions           | `AnthropicCompaction()` + `MemoryTool()`       |
| Multi-provider deployment (Anthropic and OpenAI in the same codebase) | `history_processors=[summarize_old(...)]`      |
| Need explicit audit of what was dropped                               | `history_processors=[token_budget_trim(...)]`  |
| Recency-bounded chat, dropping early turns is fine                    | `history_processors=[sliding_window(...)]`     |

Two mental hooks for the difference between compaction and memory:

- **Compaction** is *subtractive* — it shrinks the running history.
- **Memory** is *additive* — the agent writes facts to a store that outlives the chat.

They compose. The "Anthropic, remember facts across sessions" row is exactly that: compaction handles the transcript, memory handles the durable facts.

## Recap

- **Multi-agent** — parent calls a specialist via a tool; pass `usage=ctx.usage` for rolled-up token counts.
- **`pydantic_graph`** — explicit branches, loops, retries, shared state. Reach for it when control flow is worth showing.
- **`agent.iter()`** — walk the agent loop node-by-node for debugging or tracing.
- **`message_history=`** — multi-turn conversations. `new_messages()` for deltas, `all_messages()` for the rolling transcript.
- **History processors** — `sliding_window` (free, lossy), `token_budget_trim` (free, smarter), `summarize_old` (one LLM call per compaction event, coherent).
- **`AnthropicCompaction()`** — server-side, Anthropic-only, cheapest path. `compaction_*` keys appear in `usage` when it fires.
- **`MemoryTool` (the working version)** — backing function tool named `memory` dispatches `view` / `create` / `str_replace` / `delete` against whatever store you wire up. Cross-conversation continuity without `message_history=`.

The next notebook covers production concerns — testing, evals, observability, hooks, fallback models, deployment surfaces.